In [ ]:
from agent_framework import (
    Agent,
    tool,
)
from agent_framework.azure import AzureOpenAIChatClient
from typing import Annotated
from dotenv import load_dotenv
from pydantic import Field
from random import randint

import sqlite3

import os
load_dotenv()

In [ ]:
# NOTE: approval_mode="never_require" is for sample brevity.
# Use "always_require" in production for user confirmation before tool execution.
@tool(approval_mode="never_require")
def get_weather(
    location: Annotated[str, Field(description="The location to get the weather for.")],
) -> str:
    """Get the weather for a given location."""
    conditions = ["sunny", "cloudy", "rainy", "stormy"]
    return f"The weather in {location} is {conditions[randint(0, 3)]} with a high of {randint(10, 30)}°C."

In [ ]:
@tool(approval_mode="never_require")
def get_database_schema() -> str:
    """Get the database schema with random example rows from all tables in products.db."""
    db_path = "data/products.db"
    
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Get all table names, excluding system tables
        cursor.execute("SELECT name FROM sqlite_master WHERE type='table' AND name NOT LIKE 'sqlite_%';")
        tables = cursor.fetchall()
        
        schema_info = "# Database Schema\n\n"
        
        for table in tables:
            table_name = table[0]
            schema_info += f"## Table: {table_name}\n"
            
            # Get column info
            cursor.execute(f"PRAGMA table_info({table_name});")
            columns = cursor.fetchall()
            
            schema_info += "### Columns:\n"
            for col in columns:
                col_name, col_type = col[1], col[2]
                schema_info += f"- `{col_name}` ({col_type})\n"
            
            # Get random sample rows (limit to 3)
            cursor.execute(f"SELECT * FROM {table_name} ORDER BY RANDOM() LIMIT 3;")
            rows = cursor.fetchall()
            
            if rows:
                schema_info += "### Example Rows:\n```\n"
                # Get column names
                col_names = [description[0] for description in cursor.description]
                schema_info += " | ".join(col_names) + "\n"
                schema_info += " | ".join(["---"] * len(col_names)) + "\n"
                
                for row in rows:
                    schema_info += " | ".join(str(val) for val in row) + "\n"
                
                schema_info += "```\n"
            
            schema_info += "\n"
        
        conn.close()
        return schema_info
        
    except Exception as e:
        return f"Error retrieving database schema: {str(e)}"
    
print(get_database_schema())

In [ ]:
@tool(approval_mode="never_require")
def get_database_schema_string():
    print("Returning enriched database schema string...")
    ENRICHED_SCHEMA = """
        DATABASE OVERVIEW

        This schema represents LED product technical documentation extracted from PDFs.
        It is designed to support hybrid querying (SQL + semantic layer) and agent-based reasoning.

        --------------------------------------------------
        TABLE: product
        --------------------------------------------------
        Core product identity and ordering information.

        Columns:
        - product_id: Unique identifier
        - commercial_name: Full product name (often encodes size, lumen, CCT, variant)
        - eoc: Philips commercial product code (european ordering code)
        - nc_12: 12NC (12-digit Philips material number, supply chain identifier)
        - eprel_registration: EU energy label registration (if available)
        - box_quantity: Units per box
        - document_name: Source PDF
        - page_num: Page where extracted
        - source_table_family: Source table type (e.g. ordering_data)

        Notes:
        - commercial_name often embeds structured info (length, lumen, CCT, variant)
        - eoc and nc_12 are key identifiers in real-world systems


        --------------------------------------------------
        TABLE: product_spec
        --------------------------------------------------
        Semi-structured specification table.

        Each row represents ONE specification entry for a product.

        Columns:
        - spec_id: Unique identifier
        - product_id: FK → product
        - spec_name: Name of specification (UNSTRUCTURED / SEMI-STANDARDIZED)
        - unit: Measurement unit (if applicable)
        - value: Direct value (string)
        - min_value: Minimum value
        - nominal: Typical value
        - max_value: Maximum value
        - condition: Condition under which value applies
        - life_value: Lifetime-related value (if applicable)
        - raw_json: Original parsed row
        - document_name: Source PDF
        - page_num: Page number
        - source_table_family: Table structure type

        IMPORTANT:
        spec_name is NOT normalized and must be interpreted.

        It can represent:
        1. Physical properties (Length, Width, Height)
        2. Electrical properties (Voltage, Current, Power)
        3. Optical properties (Luminous flux, CCT, CRI)
        4. Environmental limits (Temperature, ESD)
        5. System configuration (modules per chain)
        6. Compliance / labeling (energy label)
        7. Performance metrics (efficacy, degradation)
        8. Ambiguous or malformed entries (requires cleaning)


        --------------------------------------------------
        TABLE: product_performance
        --------------------------------------------------
        Structured photometric and electrical performance data.

        Columns:
        - luminous_flux: Output (lumens)
        - efficacy: lm/W
        - input_current_ma: Current
        - operation_point: Operating mode
        - temperature_case: Tc condition
        - color_code: e.g. 830, 840, 850

        Notes:
        - This table is structured and preferred over product_spec for performance queries


        --------------------------------------------------
        TABLE: product_lifetime
        --------------------------------------------------
        Lifetime and degradation metrics.

        Columns:
        - L70, L80, L90 metrics with B10/B20/B50 variants
        - operation_point
        - temperature_case

        Notes:
        - Encodes lumen maintenance over time
        - Used for reliability and durability analysis


        ==================================================
        SPEC_NAME SEMANTIC NORMALIZATION
        ==================================================

        The following cleaned categories define how spec_name should be interpreted.

        (Filtered: removed numeric ranges, and product-name-like entries)

        --------------------------------------------------
        THERMAL / ENVIRONMENTAL
        --------------------------------------------------
        - Ambient temperature
        - Storage temperature
        - Case temperature (Tc-max)
        - Tc (case temperature at Tc point)

        --------------------------------------------------
        OPTICAL PROPERTIES
        --------------------------------------------------
        - Luminous flux
        - Correlated color temperature (CCT)
        - Color consistency
        - Color coordinates (CIEx, CIEy)
        - CRI
        - R9
        - Radiation angle
        - Photometric code

        --------------------------------------------------
        ELECTRICAL PROPERTIES
        --------------------------------------------------
        - Forward voltage
        - Working voltage
        - Maximum current through chain
        - Current through the LED module (I-max)
        - Power consumption
        - Power at rated Vf-max and I-max
        - Thermal power
        - Voltage strength

        --------------------------------------------------
        EFFICIENCY / PERFORMANCE
        --------------------------------------------------
        - Efficacy
        - Module efficacy

        --------------------------------------------------
        LIFETIME / RELIABILITY
        --------------------------------------------------
        - M70F50 life
        - M70F50 nominal
        - M80F50 @ nominal
        - M80F50 @Tc life
        - Δu'v' at 6000 hours

        --------------------------------------------------
        MECHANICAL / DIMENSIONS
        --------------------------------------------------
        - Length
        - Width
        - Height PCB
        - Height Total
        - Height excl. connector
        - Height incl. connector
        - Height total
        - Height with connector
        - Product mass

        --------------------------------------------------
        SYSTEM CONFIGURATION
        --------------------------------------------------
        - Number of modules per chain
        - Number of modules in series per chain
        - Number of modules in parallel
        - Number of modules in parallel per chain

        --------------------------------------------------
        SAFETY / COMPLIANCE
        --------------------------------------------------
        - Photobiological safety
        - ESD (air)
        - ESD (direct contact)
        - Energy efficiency label
        - Energy label

        --------------------------------------------------
        WIRING / INSTALLATION
        --------------------------------------------------
        - Input wire cross-section
        - Input wire strip length

        --------------------------------------------------
        MISC / DOMAIN-SPECIFIC
        --------------------------------------------------
        - application
    """
    return ENRICHED_SCHEMA


In [ ]:
@tool(approval_mode="never_require")
def execute_database_query(
    query: Annotated[str, Field(description="The SQL query to execute against products.db")],
) -> str:
    """Execute a SQL query against the products.db database and return results or error."""
    db_path = "data/products.db"
    print(f"Executing query:\n{query}\n")
    try:
        conn = sqlite3.connect(db_path)
        cursor = conn.cursor()
        
        # Execute the query
        cursor.execute(query)
        
        # Get results
        rows = cursor.fetchall()
        col_names = [description[0] for description in cursor.description]
        
        if not rows:
            result = "Query executed successfully but returned no results."
        else:
            # Format results as a readable table
            result = " | ".join(col_names) + "\n"
            result += " | ".join(["---"] * len(col_names)) + "\n"
            
            for row in rows:
                result += " | ".join(str(val) for val in row) + "\n"
        
        conn.close()
        return result
        
    except Exception as e:
        return f"Error executing query: {str(e)}"

In [ ]:
import re
from typing import Annotated

@tool(approval_mode="never_require")
def tokenize_search_term(
    search_term: Annotated[str, Field(description="Raw search term, e.g. 'fortimo HV5'")]
) -> list[str]:
    """Split a search term by whitespace and normalize to lowercase tokens."""
    tokens = [t.lower() for t in re.split(r"\s+", search_term.strip()) if t]
    print(f"Tokenized search term '{search_term}' into tokens: {tokens}")
    return tokens

@tool(approval_mode="never_require")
def build_like_clause(
    tokens: Annotated[list[str], Field(description="Token list, e.g. ['fortimo', 'hv5']")]
) -> str:
    """
    Build a SQLite-compatible LIKE clause from tokens.
    Example output: LOWER(product_name) LIKE '%fortimo%' AND LOWER(product_name) LIKE '%hv5%'
    """
    if not tokens:
        return "1=1"

    # Basic quote escaping for SQL string literals
    safe_tokens = [t.replace("'", "''").lower() for t in tokens]

    result = " AND ".join(
        [f"LOWER(product_name) LIKE '%{t}%'" for t in safe_tokens]
    )
    print(f"Built LIKE clause: {result}")
    return result

In [ ]:
from openai import AzureOpenAI
import json

# Initialize Azure OpenAI client for tokenization
tokenizer_client = AzureOpenAI(
    api_key=os.environ["AZURE_OPENAI_API_KEY"],
    api_version="2024-02-01",
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"]
)

@tool(approval_mode="never_require")
def tokenize_search_text_with_llm(
    user_text: Annotated[str, Field(description="Raw user search text to be tokenized into product and attribute")],
) -> str:
    """
    Uses LLM to extract and normalize product name and attribute from user text.
    Returns a normalized query string in format: '{product} {attribute}'.
    
    Example:
    Input: "what's the power consumption of fortimo hv5?"
    Output: "fortimo hv5 power"
    """
    
    system_prompt = """You are a search query normalizer for LED product specifications.

Your task: Extract the PRODUCT name and ATTRIBUTE from user text.

Products are LED products like "fortimo hv5", "xitanium", "certadrive". 
Attributes are something that describes a property, specification, or usage/application of the product.

Rules:
1. Extract ONE product name (prefer known products, but accept others if mentioned)
2. Extract ONE attribute (prefer known attributes, use closest match)
3. Normalize to lowercase, and alphabetically sort products and attributes if multiple mentioned
4. Return ONLY valid JSON: {"product": "...", "attribute": "..."}
5. If product unclear, use empty string ""
6. If attribute unclear, use empty string ""

Examples:

User: "what's the power of fortimo hv5?"
Output: {"product": "fortimo hv5", "attribute": "power"}

User: "show me the light I can install in the reading room."
Output: {"product": "", "attribute": "reading room light"}
"""
    
    try:
        print(f"[Tokenizer] Processing user text: '{user_text}'")
        
        response = tokenizer_client.chat.completions.create(
            model="gpt-4.1",  # deployment name
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_text}
            ],
            temperature=0.0,  # Deterministic
            max_tokens=100,
            seed=42  # Additional determinism control
        )
        
        result_text = response.choices[0].message.content.strip()
        print(f"[Tokenizer] LLM response: {result_text}")
        
        # Parse JSON response
        parsed = json.loads(result_text)
        product = parsed.get("product", "").strip()
        attribute = parsed.get("attribute", "").strip()
        
        # Build query string
        query_parts = [p for p in [product, attribute] if p]
        query = " ".join(query_parts)
        
        print(f"[Tokenizer] Extracted - Product: '{product}', Attribute: '{attribute}'")
        print(f"[Tokenizer] Generated query: '{query}'")
        
        return query
        
    except json.JSONDecodeError as e:
        print(f"[Tokenizer] JSON parse error: {e}")
        print(f"[Tokenizer] Raw response: {result_text}")
        # Fallback: return original text
        return user_text
    except Exception as e:
        print(f"[Tokenizer] Error: {str(e)}")
        return user_text

In [ ]:
print(execute_database_query("SELECT spec_name, value, nominal, min_value, max_value, unit, condition FROM product_spec WHERE product_id = 12 AND LOWER(spec_name) LIKE '%photobiological safety%'"))

In [ ]:
instruction="""
You are an assistant that queries a SQLite database containing lighting product specifications.

Follow these rules strictly:

1. ALWAYS call `get_database_schema` first if schema is not already known.
2. NEVER assume table or column names. Only use names returned by the schema.
3. ALWAYS generate valid SQLite SQL.

SEARCH STRATEGY:

4. If the user query involves product name or free-text matching:

   * Call `tokenize_search_term`
   * Then call `build_like_clause`
   * Use the generated clause to search relevant columns (e.g. product_name, spec_name)
   * Use OR-based matching unless exact match is clearly required

5. Use LIKE-based matching ONLY for:

   * product identification
   * fuzzy specification name matching

6. Once relevant products or specifications are identified:

   * Extract product IDs or canonical fields
   * Use those identifiers in subsequent queries instead of repeating LIKE

QUERY RULES:

7. ALWAYS include a LIMIT clause unless aggregation is requested.
8. Prefer simple queries over complex ones.
9. Use explicit conditions (no implicit joins).
10. If a query returns an error or empty result:

    * revise the query using available information
    * avoid repeating the same mistake

OUTPUT RULES:

11. First generate the SQL query.
12. Then call `execute_database_query` with that query.
13. Do NOT return SQL without executing it.

FAILURE HANDLING:

14. If the request cannot be answered with available schema:

    * explain what is missing
    * do NOT hallucinate data

Be precise and deterministic.

"""
client = AzureOpenAIChatClient(
    endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    deployment_name='gpt-4.1',
    api_key=os.environ["AZURE_OPENAI_API_KEY"]
)

agent = Agent(
    name="SQL_Agent",
    instructions=instruction,
    client=client,
    tools=[get_database_schema_string, tokenize_search_term, build_like_clause, execute_database_query],
    default_options={
      "tool_choice": "required",
      "temperature": 0.0
    }
)

<h3> Tool for Index data </h3>
<p><strong>Note:</strong> FAISS search configured for 100% deterministic results:</p>
<ul>
<li>Random seeds set (Python, NumPy, PyTorch)</li>
<li>Model in evaluation mode (no dropout/stochastic layers)</li>
<li>Consistent encoding parameters (normalize_embeddings=False, fixed batch_size)</li>
<li>Same settings used during index creation in data_ingestion.ipynb</li>
</ul>

In [ ]:
import json
import pickle
from pathlib import Path
import faiss
import numpy as np
import random
import torch
from sentence_transformers import SentenceTransformer

# Set random seeds for deterministic behavior
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

_embedding_model_cache = {}


def _get_embedding_model(model_name: str) -> SentenceTransformer:
    if model_name not in _embedding_model_cache:
        model = SentenceTransformer(model_name)
        model.eval()  # Set to evaluation mode for deterministic inference
        _embedding_model_cache[model_name] = model
    return _embedding_model_cache[model_name]


@tool(approval_mode="never_require")
def search_local_faiss_index(
    search_term: Annotated[str, Field(description="Text query to search in local FAISS index")],
    top_n: Annotated[int, Field(description="Number of top results to return", ge=1, le=50)] = 5
) -> str:
    """
    Search local FAISS index and return top-n matches with product purpose/design context.
    """
    try:
        print(f"Searching FAISS index for query: '{search_term}' with top_n={top_n}")
        # Paths to the FAISS index created by data_ingestion.ipynb
        index_dir = "data/faiss_index"
        embedding_model_name = "sentence-transformers/all-MiniLM-L6-v2"

        index_dir_path = Path(index_dir)
        index_path = index_dir_path / "faiss.index"
        metadata_path = index_dir_path / "metadata.pkl"

        if not index_path.exists():
            return f"Error: FAISS index file not found at {index_path}"
        
        if not metadata_path.exists():
            return f"Error: Metadata file not found at {metadata_path}"

        # Load FAISS index
        index = faiss.read_index(str(index_path))

        # Load metadata and documents
        with open(metadata_path, "rb") as f:
            data = pickle.load(f)
            documents = data["documents"]
            metadata = data["metadata"]

        # Generate query embedding with deterministic settings
        model = _get_embedding_model(embedding_model_name)
        query_vec = model.encode(
            [search_term], 
            convert_to_numpy=True,
            show_progress_bar=False,
            normalize_embeddings=False,  
            batch_size=1 
        )
        query_vec = np.array(query_vec, dtype=np.float32)

        # Guard against embedding dimension mismatch
        if query_vec.shape[1] != index.d:
            return (
                f"Error: embedding dimension mismatch. "
                f"query_dim={query_vec.shape[1]}, index_dim={index.d}, "
                f"model={embedding_model_name}"
            )

        # Search FAISS index
        distances, indices = index.search(query_vec, top_n)

        results = []
        for rank, (idx, distance) in enumerate(zip(indices[0], distances[0]), start=1):
            if idx < 0 or idx >= len(documents):
                continue

            results.append(
                {
                    "rank": rank,
                    "score": float(distance),  # L2 distance: lower is better
                    "id": int(idx),
                    "text": documents[idx],
                    "metadata": metadata[idx],
                }
            )

        if not results:
            print("No results found in FAISS index search.")
            return "No results found."

        json_results = json.dumps(results, ensure_ascii=False, indent=2)
        print(f"FAISS search results:\n{json_results}")
        return json_results

    except Exception as e:
        return f"Error searching FAISS index: {e}"


In [ ]:
result = search_local_faiss_index("kitchen workbench optimal lighting", top_n=8)

In [ ]:
print(result)

<h3> Create SQL Agent Tool Wrapper </h3>

In [ ]:
# Rename the SQL agent to sql_agent for clarity
sql_agent = agent

@tool(approval_mode="never_require")
async def query_product_database(
    question: Annotated[str, Field(description="A question about product specifications, filtering, or detailed technical data")]
) -> str:
    """
    Query the product database for detailed specifications, filtering, and structured data.
    
    Use this tool when you need to:
    - Get specific technical specifications (voltage, current, dimensions, CCT values, etc.)
    - Filter products by criteria (e.g., products with > 2000 lumens)
    - Find exact values from structured data
    - Compare specifications across products
    - Look up by product name or model number
    
    The tool uses an intelligent SQL agent that can query tables:
    - product: product names, ordering codes (eoc, nc_12)
    - product_spec: specifications (semi-structured)
    - product_performance: luminous flux, efficacy, current
    - product_lifetime: L70/L80/L90 lifetime metrics
    """
    try:
        print(f"[SQL Agent Tool] Processing question: {question}")
        sql_session = sql_agent.create_session()
        response = await sql_agent.run(question, session=sql_session)
        result = str(response)
        print(f"[SQL Agent Tool] Response: {result[:200]}...")
        return result
    except Exception as e:
        return f"Error querying database: {str(e)}"

<h3> Create Parent Orchestrator Agent </h3>

In [ ]:
orchestrator_instructions = """
You are an intelligent product assistant that helps users find LED lighting products and their specifications.

You have access to TWO complementary data sources:

1. **tokenize_search_text_with_llm**: LLM-powered search term extraction and normalization
   - Use for: Extracting product names and attributes from free-text queries
   - Use for: Generating normalized search queries for FAISS index
   - Returns: Normalized query string (e.g. "fortimo hv5 power")
2. **search_local_faiss_index**: Purpose & design descriptions (semantic/text search)
   - Use for: "What product is good for X application?"
   - Use for: "Which products are designed for retail/office/outdoor?"
   - Use for: Understanding product purpose, use cases, recommendations
   - Returns: Text descriptions with product names and context

3. **query_product_database**: Detailed specifications (structured SQL data)
   - Use for: "What is the CCT of product X?"
   - Use for: "Find products with >2000 lumens"
   - Use for: Exact specifications, filtering, comparisons
   - Returns: Structured data (specs, performance, lifetime)

DECISION STRATEGY:

A. For DISCOVERY / RECOMMENDATION queries:
   → Start with search_local_faiss_index (top 5-10 results)
   → Extract product names from results
   → THEN query_product_database for detailed specs of those products

B. For SPECIFICATION queries (when product name/family is known):
   → Use query_product_database directly

C. For COMPLEX queries (e.g., "best product for retail with >2500 lumens"):
   → First: search_local_faiss_index to find retail-suitable products
   → Second: query_product_database to filter by lumen requirement

RESPONSE RULES:

1. Always explain your search strategy briefly
2. When using index results, show product names found
3. When using database, show the actual specification values
4. Combine results intelligently - don't just dump raw data
5. If one tool doesn't give complete answer, try the other or combine both
6. Be conversational but precise

EXAMPLES:

User: "Which light is good for living room?"
→ search_local_faiss_index with "living room residential ambient lighting"
→ Return product recommendations with context

User: "What is the CCT of Fortimo LED HV5?"
→ query_product_database to find CCT specification for that product

User: "I need a bright light for retail, at least 2500 lumens"
→ search_local_faiss_index for "retail lighting"
→ query_product_database to filter products by luminous_flux >= 2500
→ Combine results and recommend matching products

Be helpful, thorough, and accurate.
"""

orchestrator_client = AzureOpenAIChatClient(
    endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    deployment_name='gpt-4.1',
    api_key=os.environ["AZURE_OPENAI_API_KEY"]
)

orchestrator_agent = Agent(
    name="Product_Assistant",
    instructions=orchestrator_instructions,
    client=orchestrator_client,
    temperature=0.0, 
    tools=[tokenize_search_text_with_llm, search_local_faiss_index, query_product_database],
    default_options={
      "tool_choice": "required",
      "temperature": 0.0
    }
)

print("Orchestrator agent created successfully!")

<h3> Test the Orchestrator Agent </h3>

In [ ]:
# Test with a discovery query
test_query = "Which products are suitable for retail applications with good efficacy?"
print(f"Testing orchestrator with: {test_query}\n")
print("="*80)

test_session = orchestrator_agent.create_session()
test_response = await orchestrator_agent.run(test_query, session=test_session)
print(f"\nOrchestrator Response:\n{test_response}")

<h3> Interactive Chat with Orchestrator Agent </h3>

In [ ]:
# Continuous chat loop with orchestrator agent (keeps context between turns)
print("Product Assistant Chat")
print("="*80)
print("Ask questions about LED products - I can search by purpose/application or get detailed specs!")
print("Type 'exit', 'quit', or 'q' to end the session.\n")

chat_history = []
session = orchestrator_agent.create_session()

while True:
    user_input = input("You: ").strip()
    if user_input.lower() in {"exit", "quit", "q"}:
        print("Session ended. Thank you!")
        break
    if not user_input:
        continue
    
    print(f"Processing your question...\n")
    print(user_input)
    chat_history.append(f"User: {user_input}")

    # Build conversational context for the agent
    convo_context = "\n".join(chat_history[-10:])  # keep last 10 turns
    prompt = (
        "Previous conversation:\n"
        f"{convo_context}\n\n"
        "Answer the user's latest question using the appropriate tools.\n"
        f"Latest question: {user_input}"
    )

    try:
        response = await orchestrator_agent.run(prompt, session=session)
        response_text = str(response)
    except Exception as e:
        response_text = f"Error: {e}"

    print(f"Assistant: {response_text}\n")
    print("-"*80 + "\n")
    chat_history.append(f"Assistant: {response_text}")